# Phase 3 — Source-Held-Out Probes

**The question.** Does the model read clauses, or does it recognise datasets? The fused
corpus stitches three annotation projects together, each with its own drafting register,
clause segmentation and label vocabulary. A model that has learned "this looks like a
CLAUDETTE row, and CLAUDETTE rows about arbitration are usually harmful" would score well
on a random split while having learned very little about arbitration.

The existing `notebooks/model_finetuning/lawgic_classifier_probe.ipynb` already showed
that source identity is *linearly decodable* from the fine-tuned encoder. That is
necessary but not sufficient evidence: an encoder can carry source information without the
heads depending on it. This notebook tests the stronger claim directly — **remove a source
from training entirely, then evaluate only on that source's rows.**

## Two probes, and why not three

| Probe | Held out | Corpus rows carrying that source |
| --- | --- | --- |
| A | CLAUDETTE | 3,182 wide rows (3,721 long-format annotation rows) |
| B | 100 ToS | 1,460 wide rows (2,048 long-format annotation rows) |
| — | ~~ToS;DR~~ | **deliberately not run** |

The row counts differ between the long and wide formats because the wide corpus is one row
per unique clause: a clause annotated with several topics by the same source collapses into
one row with several active topic cells. The holdout operates on wide rows, so those are
the numbers reported.

**Why there is no ToS;DR holdout.** ToS;DR supplies 21,949 of 26,479 rows — about 83% of
the corpus and roughly 88% of training rows once the split is applied. Removing it would
leave ~2,500 training clauses. A score collapse under that condition is uninterpretable:
it would be perfectly consistent with "the model only recognised ToS;DR" *and* with "no
model learns 44-way multi-label legal topic detection from 2,500 examples". The probe
would be confounded with data starvation and would answer neither question. The two
smaller sources can be removed while leaving the training regime broadly intact, which is
what makes their results readable.

## Protocol

Legal-BERT, seed 42, dual-head, identical to Phase 2 in every other respect — same
persisted seed-42 split, same hyperparameters, same losses, same early stopping, same
degenerate-model assertion. The holdout removes the source's rows from **train and
validation** and restricts the **test** set to exactly those rows.

## Masking: score only what the held-out source actually supervised

This is the part that decides whether the probe means anything.

The supervision mask is source-aware: a row annotated by CLAUDETTE has observed cells only
for the topics CLAUDETTE's label vocabulary covers; every other cell is *unknown* and
contributes zero loss. If a held-out CLAUDETTE row is scored across all 44 topics, most of
the score comes from cells CLAUDETTE never labelled — the model is being graded against
the shape of the mask, not against comprehension of the clause.

`source_supervision_mask()` in `scripts/lawgic_train_matrix.py` rebuilds, per row, the set
of topic cells the held-out source itself asserted (read back out of `native_annotations`,
which records `source_dataset` and `lawgic_topic_id` for every annotation) and intersects
it with the row's existing supervision mask. Scoring runs over that intersection only.

The in-distribution baseline is computed on **exactly the same rows and the same cell
mask**, taken from the Phase 2 legal-bert/seed-42 run's stored logits. Without that
restriction the "retained ratio" would be comparing two different denominators and would
be meaningless.

In [1]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    sentinel = Path("generated_files/lawgic_taxonomy/lawgic_multihead_wide.csv")
    for candidate in (start, *start.parents):
        if (candidate / sentinel).exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the lawgic repository.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

import json

import numpy as np
import pandas as pd

import lawgic_eval_core as core
import lawgic_train_matrix as tm

pd.set_option("display.width", 160)

core.persist_split()
corpus = core.load_corpus()
frames = core.split_frames(corpus)
assert {k: len(v) for k, v in frames.items()} == core.EXPECTED_SPLIT_ROWS

HOLDOUT_SOURCES = ["claudette", "100_tos"]
BASELINE_RUN = tm.RunConfig(encoder_name=tm.ENCODERS[0], seed=42, heads="dual").run_id

for source in HOLDOUT_SOURCES + ["tos_dr"]:
    corpus_rows = int(tm.source_row_mask(corpus, source).sum())
    train_rows = int(tm.source_row_mask(frames["train"], source).sum())
    test_rows = int(tm.source_row_mask(frames["test"], source).sum())
    print(f"{source:>10}: corpus {corpus_rows:>6,} | train {train_rows:>6,} "
          f"({train_rows / len(frames['train']):.1%}) | test {test_rows:>5,}")

print(f"\nPhase 2 baseline run required: {BASELINE_RUN}")

 claudette: corpus  3,182 | train  2,528 (11.9%) | test   324
   100_tos: corpus  1,460 | train  1,175 (5.5%) | test   141
    tos_dr: corpus 21,949 | train 17,559 (82.9%) | test 2,199

Phase 2 baseline run required: legal-bert-base-uncased__seed42__dual


## MANUAL STEP — prerequisites

1. **Run Phase 2 first.** This notebook reads
   `generated_files/lawgic_taxonomy/runs/legal-bert-base-uncased__seed42__dual/test_logits.npz`
   for the in-distribution baseline. Without it there is nothing to compare against.
2. **GPU.** Two more full fine-tunes, same order of magnitude per run as a Phase 2 run
   (slightly faster — training sets are smaller by the held-out source).
3. No downloads or credentials: legal-bert is already local.

In [2]:
baseline_path = tm.RUNS_DIR / BASELINE_RUN / "test_logits.npz"
if not baseline_path.exists():
    raise FileNotFoundError(
        f"{baseline_path} missing. Run 02_multiseed_encoder_runs.ipynb (at least the "
        f"legal-bert/seed42/dual config) before this notebook."
    )
print(f"Baseline logits found: {baseline_path}")

Baseline logits found: C:\Users\Enrique\Coding Projects\Thesis\lawgic\generated_files\lawgic_taxonomy\runs\legal-bert-base-uncased__seed42__dual\test_logits.npz


## Run the two probes

Resumable in the same way as Phase 2: completed probes are skipped.

In [3]:
FORCE_RERUN = False

probe_configs = [
    tm.RunConfig(encoder_name=tm.ENCODERS[0], seed=42, heads="dual", holdout_source=source)
    for source in HOLDOUT_SOURCES
]

probe_records = []
for config in probe_configs:
    target = tm.RUNS_DIR / config.run_id / "metrics.json"
    if target.exists() and not FORCE_RERUN:
        print(f"skip {config.run_id} (already complete)")
        probe_records.append(json.loads(target.read_text()))
        continue
    print(f"running {config.run_id} ...")
    probe_records.append(tm.run_config(config))

display(pd.DataFrame(probe_records)[
    ["run_id", "train_rows", "val_rows", "test_rows", "wall_seconds", *core.HEADLINE_METRICS]
])

running legal-bert-base-uncased__seed42__dual__holdout-claudette ...


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Topic Macro F1,Topic Micro F1,Topic Weighted F1,Topic Observed Positions,Risk Accuracy,Risk Macro F1,Risk Weighted F1,Risk Valid Rows
1,0.686100,0.693199,0.043595,0.156237,0.127632,95754.000000,0.785591,0.778418,0.781248,2318.000000
2,0.548300,0.570489,0.373230,0.594494,0.532293,95754.000000,0.845557,0.842307,0.844536,2318.000000
3,0.447400,0.599906,0.562849,0.744936,0.716438,95754.000000,0.839517,0.835601,0.839249,2318.000000
4,0.331800,0.626591,0.640152,0.790384,0.780176,95754.000000,0.846851,0.844435,0.846545,2318.000000
5,0.248200,0.760401,0.657905,0.815456,0.803791,95754.000000,0.850302,0.847016,0.849959,2318.000000
6,0.138000,0.856335,0.684565,0.826607,0.818462,95754.000000,0.857204,0.853574,0.856764,2318.000000
7,0.101500,0.914346,0.705388,0.826838,0.821016,95754.000000,0.848576,0.844460,0.847789,2318.000000
8,0.044900,0.957626,0.724620,0.836797,0.832946,95754.000000,0.861950,0.858810,0.862017,2318.000000
9,0.115100,0.993424,0.737383,0.835534,0.832772,95754.000000,0.856342,0.853499,0.856251,2318.000000
10,0.051400,1.040153,0.753562,0.829797,0.828310,95754.000000,0.861519,0.858369,0.860948,2318.000000


[legal-bert-base-uncased__seed42__dual__holdout-claudette] 39.3 min | topic_macro_f1=0.5805 topic_micro_f1=0.6264 risk_accuracy=0.4599 risk_macro_f1=0.3614
running legal-bert-base-uncased__seed42__dual__holdout-100_tos ...


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Topic Macro F1,Topic Micro F1,Topic Weighted F1,Topic Observed Positions,Risk Accuracy,Risk Macro F1,Risk Weighted F1,Risk Valid Rows
1,0.614100,0.849103,0.086818,0.274882,0.217625,92258.000000,0.816294,0.799159,0.812814,2504.000000
2,0.560900,0.601022,0.436796,0.659733,0.603573,92258.000000,0.848243,0.841337,0.847981,2504.000000
3,0.430700,0.591784,0.608810,0.774715,0.753675,92258.000000,0.859425,0.852359,0.859526,2504.000000
4,0.296500,0.690853,0.668719,0.812317,0.801666,92258.000000,0.851038,0.842267,0.851030,2504.000000
5,0.200900,0.729212,0.694369,0.826828,0.817691,92258.000000,0.862220,0.854906,0.862529,2504.000000
6,0.133400,0.828332,0.709865,0.833297,0.830973,92258.000000,0.855831,0.848677,0.854891,2504.000000
7,0.170000,0.950799,0.723906,0.845937,0.842834,92258.000000,0.859425,0.853676,0.859550,2504.000000
8,0.094300,0.961102,0.735244,0.855598,0.853579,92258.000000,0.863419,0.857577,0.863170,2504.000000
9,0.088700,0.941349,0.741741,0.849864,0.848616,92258.000000,0.866613,0.859767,0.865956,2504.000000
10,0.076500,1.038351,0.751883,0.852028,0.850408,92258.000000,0.864217,0.856624,0.863500,2504.000000


[legal-bert-base-uncased__seed42__dual__holdout-100_tos] 39.0 min | topic_macro_f1=0.5197 topic_micro_f1=0.6414 risk_accuracy=0.4113 risk_macro_f1=0.3578


,run_id,train_rows,val_rows,test_rows,wall_seconds,topic_macro_f1,topic_micro_f1,risk_accuracy,risk_macro_f1
0,legal-bert-base-uncased__seed42__dual__holdout...,18655,2318,324,2357.185558,0.580522,0.626374,0.459877,0.361433
1,legal-bert-base-uncased__seed42__dual__holdout...,20008,2504,141,2341.405965,0.519682,0.641379,0.411348,0.357797


## In-distribution baseline on the same rows and same cells

The Phase 2 run scored the full 2,648-row test split. Here it is re-scored on the subset
of test rows belonging to the held-out source, under that source's own supervision mask —
the identical evaluation surface the probe faces.

In [4]:
baseline = np.load(baseline_path)
baseline_row_ids = baseline["row_id"]
row_position = {int(r): i for i, r in enumerate(baseline_row_ids)}

test_frame = frames["test"]


def in_distribution_metrics(source: str) -> dict:
    """Phase 2 legal-bert/seed42 performance restricted to `source`'s test cells."""
    source_rows = test_frame[tm.source_row_mask(test_frame, source)].copy()
    positions = np.array([row_position[int(r)] for r in source_rows["row_id"]])

    arrays = core.label_arrays(source_rows)
    arrays["label_masks"] = tm.source_supervision_mask(source_rows, source)

    return core.all_metrics(
        baseline["topic_logits"][positions],
        baseline["harm_logits"][positions],
        arrays,
    )


rows = []
for source, record in zip(HOLDOUT_SOURCES, probe_records):
    indist = in_distribution_metrics(source)
    rows.append({"source": source, "condition": "in-distribution (Phase 2)", **indist})
    rows.append({"source": source, "condition": "held-out (Phase 3)",
                 **{k: record[k] for k in indist if k in record}})

comparison = pd.DataFrame(rows)
display(comparison)

,source,condition,topic_macro_f1,topic_micro_f1,topic_weighted_f1,topic_observed_positions,risk_accuracy,risk_macro_f1,risk_weighted_f1,risk_valid_rows,rows
0,claudette,in-distribution (Phase 2),0.894133,0.898678,0.897062,375.0,0.771605,0.695617,0.782470,324.0,324.0
1,claudette,held-out (Phase 3),0.580522,0.626374,0.548659,375.0,0.459877,0.361433,0.500790,324.0,324.0
2,100_tos,in-distribution (Phase 2),0.791121,0.844575,0.805532,197.0,0.560284,0.489498,0.569303,141.0,141.0
3,100_tos,held-out (Phase 3),0.519682,0.641379,0.533332,197.0,0.411348,0.357797,0.424906,141.0,141.0


## Retained-performance ratio

`held-out / in-distribution`, per metric. Read it as: **what fraction of its ability does
the model keep when it has never seen a single clause from this source?**

Interpretation guide for the manuscript — state the reading you adopt rather than letting
the number speak for itself:

- **near 1.0** — the model generalises across annotation projects; performance is not an
  artifact of source recognition.
- **materially below 1.0** — part of the headline score depends on having seen that
  source's register during training. That is a real limitation of the fused-corpus design,
  not necessarily a modelling failure.
- **The supervised-cell counts matter.** With few observed cells the ratio is noisy; the
  `observed_cells` column below is there so the ratio is never read without its
  denominator.

In [5]:
metrics_for_ratio = list(core.HEADLINE_METRICS)

ratio_rows = []
for source in HOLDOUT_SOURCES:
    indist = comparison[(comparison["source"] == source) & (comparison["condition"].str.startswith("in-"))].iloc[0]
    held = comparison[(comparison["source"] == source) & (comparison["condition"].str.startswith("held"))].iloc[0]
    for metric in metrics_for_ratio:
        ratio_rows.append({
            "Source": source,
            "Metric": metric,
            "In-distribution": float(indist[metric]),
            "Held-out": float(held[metric]),
            "Retained ratio": float(held[metric]) / float(indist[metric]) if indist[metric] else float("nan"),
            "Test rows": int(held["rows"]),
            "Observed cells": int(indist["topic_observed_positions"]),
        })

probe_table = pd.DataFrame(ratio_rows)
display(probe_table)

core.write_outputs(
    probe_table,
    "phase3_source_holdout",
    caption=(
        "Source-held-out probes. Each probe retrains Legal-BERT (seed 42, dual-head, "
        "identical protocol) with one source removed from train and validation, then "
        "evaluates only on that source's test rows and only on the topic cells that "
        "source itself supervised. The in-distribution column is the Phase 2 "
        "legal-bert/seed-42 run scored on the identical rows and cells. No ToS;DR probe is "
        "reported: it would remove ~88\\% of training rows, confounding source recognition "
        "with data starvation."
    ),
    label="tab:source-holdout",
)

,Source,Metric,In-distribution,Held-out,Retained ratio,Test rows,Observed cells
0,claudette,topic_macro_f1,0.894133,0.580522,0.649257,324,375
1,claudette,topic_micro_f1,0.898678,0.626374,0.696994,324,375
2,claudette,risk_accuracy,0.771605,0.459877,0.596000,324,375
3,claudette,risk_macro_f1,0.695617,0.361433,0.519587,324,375
4,100_tos,topic_macro_f1,0.791121,0.519682,0.656893,141,197
5,100_tos,topic_micro_f1,0.844575,0.641379,0.759411,141,197
6,100_tos,risk_accuracy,0.560284,0.411348,0.734177,141,197
7,100_tos,risk_macro_f1,0.489498,0.357797,0.730948,141,197


(WindowsPath('C:/Users/Enrique/Coding Projects/Thesis/lawgic/generated_files/lawgic_taxonomy/evaluation/phase3_source_holdout.csv'),
 WindowsPath('C:/Users/Enrique/Coding Projects/Thesis/lawgic/generated_files/lawgic_taxonomy/evaluation/phase3_source_holdout.tex'))

### Per-topic detail (optional)

Which topics survive the holdout and which collapse is usually more informative than the
aggregate. Topics with a handful of observed cells will swing wildly — read the `observed`
column before drawing any conclusion from a single row.

In [6]:
for config in probe_configs:
    path = tm.RUNS_DIR / config.run_id / "per_topic.csv"
    if not path.exists():
        continue
    table = pd.read_csv(path)
    table = table[table["observed"] > 0].sort_values("f1", ascending=False)
    print(f"\n=== {config.run_id} ===")
    display(table.head(20))


=== legal-bert-base-uncased__seed42__dual__holdout-claudette ===


,topic_id,precision,recall,f1,support,observed
6,account_termination,1.00,0.893939,0.944000,66,66
7,content_removal,1.00,0.769231,0.869565,26,26
2,mandatory_arbitration,1.00,0.761905,0.864865,21,21
0,choice_of_law,1.00,0.727273,0.842105,22,22
1,choice_of_forum,1.00,0.720000,0.837209,25,25
3,class_action_waiver,1.00,0.714286,0.833333,21,21
10,macro avg,0.80,0.494940,0.580522,375,375
11,weighted avg,0.88,0.456000,0.548659,375,375
5,contract_changes,1.00,0.181818,0.307692,44,44
4,limitation_of_liability,1.00,0.180952,0.306452,105,105



=== legal-bert-base-uncased__seed42__dual__holdout-100_tos ===


,topic_id,precision,recall,f1,support,observed
1,choice_of_forum,1.000000,1.000000,1.000000,8,8
2,mandatory_arbitration,1.000000,1.000000,1.000000,2,2
3,class_action_waiver,1.000000,1.000000,1.000000,2,2
7,limitation_period,1.000000,1.000000,1.000000,1,1
11,account_termination,1.000000,1.000000,1.000000,16,16
6,warranty_disclaimer,1.000000,1.000000,1.000000,7,7
0,choice_of_law,1.000000,0.900000,0.947368,10,10
13,content_removal,1.000000,0.900000,0.947368,10,10
14,copyright_license,1.000000,0.875000,0.933333,8,8
4,limitation_of_liability,1.000000,0.666667,0.800000,12,12
